# Debug Loader Notebook (All CSV Files)

This notebook is the notebook version of `load_debug.py`.

Purpose:
- load all CSV files from the folder into Delta tables
- use a safer parse strategy for multiline/problematic CSV files
- compare parsed records vs written Delta records
- capture malformed records in a dedicated log table

## 1) Imports

This section imports Spark functions, schemas, and Fabric utilities used in the loader.

In [13]:
# run only if you need to clean the table log - oterwise keep the history of past loading
spark.sql("TRUNCATE TABLE bad_records")

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 15, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint]

In [5]:
import os
import datetime
from pyspark.sql.functions import col, concat_ws, current_timestamp, lit, monotonically_increasing_id
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from notebookutils import mssparkutils

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 7, Finished, Available, Finished, False)

## 2) Input File Discovery

This section points to your CSV folder and builds a list of CSV files to process.

In [6]:
# Specify the folder containing CSV files
csv_folder = "Files/csv/csv"

# Get a list of all files in the folder
files = mssparkutils.fs.ls(csv_folder)

# Filter for CSV files
csv_files = [file.path for file in files if file.name.endswith(".csv")]

print(f"CSV files found: {len(csv_files)}")

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 8, Finished, Available, Finished, False)

CSV files found: 68


## 3) Log Schemas

This section defines two schemas:
- `log_schema`: one summary row per file
- `malformed_schema`: one row per malformed record

In [7]:
log_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("row_amount_file", IntegerType(), True),
    StructField("row_amount_table", IntegerType(), True),
    StructField("row_amount_diff", IntegerType(), True),
    StructField("load_timestamp_utc_bronze", TimestampType(), True)
])

malformed_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("record_number", IntegerType(), True),
    StructField("entire_record", StringType(), True),
    StructField("explanation", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 9, Finished, Available, Finished, False)

## 4) Initialize Runtime Objects

This section creates empty DataFrames for logs and initializes counters.

In [8]:
df_log = spark.createDataFrame([], schema=log_schema)
df_mal_log = spark.createDataFrame([], schema=malformed_schema)

warning_count = 0
csv_count = len(csv_files)
file_processed_count = 0

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 10, Finished, Available, Finished, False)

## 5) Main Processing Loop

For each CSV file, this loop:
- builds a table name
- reads with a permissive parser
- splits well-formed and malformed rows
- writes well-formed data to Delta
- appends summary/malformed logs

Key change vs your original loader:
- validates using parsed records, not physical text line count

In [9]:
for csv_file in csv_files:
    file_processed_count += 1
    table_name = os.path.splitext(os.path.basename(csv_file))[0]
    table_name = table_name.replace(".", "_")

    print(f"{table_name}.csv is being processed. Remaining files to process: {csv_count - file_processed_count}")

    # Informational only for multiline CSVs (not used for validation).
    physical_line_count = spark.read.text(csv_file).count() - 1
    print(f"{table_name} physical line count (minus header): {physical_line_count:,}".replace(",", "'"))

    # Parse once in PERMISSIVE mode and attempt to capture malformed rows.
    df_raw = (
        spark.read.format("csv")
        .option("delimiter", ",")
        .option("header", True)
        .option("encoding", "UTF-8")
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load(csv_file)
        .cache()
    )

    # Separate parsed records into well-formed and malformed sets.
    if "_corrupt_record" in df_raw.columns:
        df_well = df_raw.filter(col("_corrupt_record").isNull()).drop("_corrupt_record")
        df_mal = df_raw.filter(col("_corrupt_record").isNotNull())
    else:
        df_well = df_raw
        df_mal = spark.createDataFrame([], df_raw.schema)

    parsed_row_count = df_well.count()
    malformed_count = df_mal.count()

    print(f"{table_name} parsed well-formed records: {parsed_row_count:,}".replace(",", "'"))
    print(f"{table_name} malformed records: {malformed_count:,}".replace(",", "'"))

    # Add technical columns before writing to Delta.
    df_well = df_well.withColumn("id", monotonically_increasing_id())
    columns_ordered = ["id"] + [c for c in df_well.columns if c != "id"]
    df_well = df_well.select(*columns_ordered)
    df_well = df_well.withColumn("load_timestamp_utc_bronze", current_timestamp())

    # Overwrite target table each run for this file.
    df_well.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

    delta_count = spark.table(table_name).count()
    print(f"{table_name} rows in delta table: {delta_count:,}".replace(",", "'"))

    # Add per-file summary log row.
    df_current_log = spark.createDataFrame([(
        table_name,
        parsed_row_count,
        delta_count,
        parsed_row_count - delta_count,
        datetime.datetime.now()
    )], schema=log_schema)
    df_log = df_log.union(df_current_log)

    # Add malformed rows to bad-record log (if present).
    if malformed_count > 0:
        if "_corrupt_record" in df_mal.columns:
            df_mal_log_entry = (
                df_mal
                .withColumn("file_name", lit(table_name))
                .withColumn("record_number", monotonically_increasing_id())
                .withColumn("entire_record", col("_corrupt_record"))
                .withColumn("explanation", lit("malformed"))
                .withColumn("timestamp", current_timestamp())
                .select("file_name", "record_number", "entire_record", "explanation", "timestamp")
            )
        else:
            df_mal_log_entry = (
                df_mal
                .withColumn("file_name", lit(table_name))
                .withColumn("record_number", monotonically_increasing_id())
                .withColumn("entire_record", concat_ws("|||", *df_mal.columns))
                .withColumn("explanation", lit("malformed"))
                .withColumn("timestamp", current_timestamp())
                .select("file_name", "record_number", "entire_record", "explanation", "timestamp")
            )

        df_mal_log = df_mal_log.union(df_mal_log_entry)

    # Validation compares parsed records to written records.
    if parsed_row_count == delta_count:
        print(f"\033[92mSUCCESS: {table_name}.csv loaded into Delta table {table_name} with matching parsed row counts\033[0m")
    else:
        print(
            f"\033[91mERROR: {table_name}.csv parsed row count {parsed_row_count:,}".replace(",", "'")
            + f" does not match Delta table row count {delta_count:,}".replace(",", "'")
            + " \033[0m"
        )
        warning_count += 1

    print("\n")

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 11, Finished, Available, Finished, False)

HumanResources_Department.csv is being processed. Remaining files to process: 67
HumanResources_Department physical line count (minus header): 16
HumanResources_Department parsed well-formed records: 16
HumanResources_Department malformed records: 0
HumanResources_Department rows in delta table: 16
SUCCESS: HumanResources_Department.csv loaded into Delta table HumanResources_Department with matching parsed row counts


HumanResources_Employee.csv is being processed. Remaining files to process: 66
HumanResources_Employee physical line count (minus header): 290
HumanResources_Employee parsed well-formed records: 290
HumanResources_Employee malformed records: 0
HumanResources_Employee rows in delta table: 290
SUCCESS: HumanResources_Employee.csv loaded into Delta table HumanResources_Employee with matching parsed row counts


HumanResources_EmployeeDepartmentHistory.csv is being processed. Remaining files to process: 65
HumanResources_EmployeeDepartmentHistory physical line count (minus h

## 6) Persist Logs and Final Status

This section writes:
- summary logs to `load_log_bronze`
- malformed records to `bad_records`

Then it prints final status and warning count.

In [10]:
# Write load summary log rows
if not df_log.rdd.isEmpty():
    df_log.write.format("delta").mode("append").saveAsTable("load_log_bronze")

if warning_count > 0:
    print("\n\033[93mALL CSV files loaded with warnings into load_log_bronze table.\033[0m")
else:
    print("\n\033[92mALL CSV files successfully loaded into load_log_bronze table.\033[0m")

# Write malformed records
if not df_mal_log.rdd.isEmpty():
    df_mal_log.write.format("delta").mode("append").saveAsTable("bad_records")
    print("\n\033[93mMalformed records have been written to the bad_records table.\033[0m")
else:
    print("\n\033[92mNo malformed records found across processed files.\033[0m")

# Final summary
if warning_count == 0:
    print("\n\033[92mNumber of tables with warnings: NONE - SUCCESS\033[0m")
else:
    print(f"\n\033[91mNumber of tables with warnings: {warning_count} -> GO CHECK LOG!\033[0m")

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 12, Finished, Available, Finished, False)


ALL CSV files successfully loaded into load_log_bronze table.

No malformed records found across processed files.

Number of tables with warnings: NONE - SUCCESS


In [11]:
df = spark.sql("SELECT * FROM LH_AdventureWorks.humanresources_jobcandidate LIMIT 1000")
display(df)

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 374483bf-085d-44f6-bd19-e5ee8ef723db)

In [12]:
# loaded tables
#spark.sql("SHOW TABLES").show(truncate=False)
# database
#spark.sql("SHOW DATABASES").show(truncate=False)
# check bad records table
df = spark.sql("SELECT * FROM LH_AdventureWorks.bad_records order by timestamp DESC LIMIT 1000")
display(df)
# check log bronze table
df1 = spark.sql("SELECT * FROM LH_AdventureWorks.load_log_bronze WHERE row_amount_diff >0 order by load_timestamp_utc_bronze DESC LIMIT 1000")
display(df1)
print(f"Incorrect rows : {df1.count()}")
df2 = spark.sql("SELECT * FROM LH_AdventureWorks.load_log_bronze WHERE row_amount_diff = 0 order by load_timestamp_utc_bronze DESC LIMIT 1000")
display(df2)
print(f"Correct rows: {df2.count()}")

StatementMeta(, ee40dded-9e38-498e-9e95-9734091be205, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 20893dcf-cc5f-4a71-8e68-df590d3f0e74)

SynapseWidget(Synapse.DataFrame, c42e41a2-08cc-45c8-a582-0eb881df270e)

Incorrect rows : 16


SynapseWidget(Synapse.DataFrame, d9f4a457-2b5b-44ff-ae39-0bdfc1808c7f)

Correct rows: 257
